In [5]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager


districts = [
    "North Goa",
    "South Goa"
]


search_types = [
    "software companies",
    "IT companies"
]


options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")



driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)



driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")



all_data = []

visited_links = set()



def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None


def extract_school_data():

    try:


        try:

            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text

        except:

            name = ""


        try:

            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text

        except:

            address = ""


        phone = ""

        try:

            phone_xpaths = [


                '//button[contains(@data-item-id,"phone")]',

                '//a[contains(@href,"tel:")]',

                '//button[contains(@aria-label,"Phone")]',

                '//button[contains(text(),"+91")]',
                '//div[contains(text(),"+91")]',
                '//span[contains(text(),"+91")]'

            ]

            for xpath in phone_xpaths:

                try:

                    elements = driver.find_elements(
                        By.XPATH,
                        xpath
                    )

                    for element in elements:

                        text = element.text.strip()

                        href = (
                            element.get_attribute("href")
                            or ""
                        )

                        if "tel:" in href:

                            phone = href.replace(
                                "tel:",
                                ""
                            ).strip()

                        elif text:

                            phone = text

                        phone = (

                            phone.replace("", "")
                            .replace(" ", "")
                            .replace("-", "")
                            .replace("(", "")
                            .replace(")", "")
                            .replace("\n", "")
                            .strip()

                        )

                        digits = ''.join(
                            filter(str.isdigit, phone)
                        )

                        if len(digits) >= 10:

                            break

                    if phone:
                        break

                except:
                    pass

        except Exception as e:

            print(f"Phone Error: {e}")



        try:

            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")

        except:

            website = ""


        try:

            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")

        except:

            rating = ""

        return {

            "Company Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None


for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Goa"
            )

            print(f"\nSearching: {query}")

    

            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

    

            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)



            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")


            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")



            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 3)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Company Name"])

                        print(
                            f"Phone: {data['Phone']}"
                        )

        

                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Company Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_df.to_csv(
                            "goa_companies_backup.csv",
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"Company Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "Company Name",
        "Google Maps URL"
    ],

    inplace=True
)

filename = "goa_companies.csv"

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Companies Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()

Opening Google Maps...
Google Maps Loaded

Searching: software companies in North Goa Goa
Search Submitted
North Goa | software companies | Scroll 1 -> 14
North Goa | software companies | Scroll 2 -> 20
North Goa | software companies | Scroll 3 -> 27
North Goa | software companies | Scroll 4 -> 34
North Goa | software companies | Scroll 5 -> 40
North Goa | software companies | Scroll 6 -> 47
North Goa | software companies | Scroll 7 -> 54
North Goa | software companies | Scroll 8 -> 60
North Goa | software companies | Scroll 9 -> 67
North Goa | software companies | Scroll 10 -> 74
North Goa | software companies | Scroll 11 -> 76
North Goa | software companies | Scroll 12 -> 76
North Goa | software companies | Scroll 13 -> 76
North Goa | software companies | Scroll 14 -> 76
North Goa | software companies | Scroll 15 -> 76
Collected 76 links
North Goa | 1/76
NineStack - A Kilowott Company
Phone: 09145246464
North Goa | 2/76
Codanto - Software Company In Goa
Phone: 09545410696
North Goa |

In [ ]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager








# =========================================================
# DRIVER
# =========================================================
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )


driver = create_driver()
wait = WebDriverWait(driver, 20)


# =========================================================
# HELPERS
# =========================================================
def search_maps(query):
    driver.get("https://www.google.com/maps")
    time.sleep(4)

    box = wait.until(EC.presence_of_element_located((By.ID, "searchboxinput")))
    box.clear()
    box.send_keys(query)
    box.send_keys(Keys.ENTER)
    time.sleep(7)


def get_listings():
    try:
        feed = wait.until(EC.presence_of_element_located((By.XPATH, '//div[@role="feed"]')))

        for _ in range(10):
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", feed)
            time.sleep(2)

        return driver.find_elements(By.XPATH, '//a[contains(@href,"/place")]')

    except:
        return []


def extract_data(state, district, category):

    def safe(xpath):
        try:
            return driver.find_element(By.XPATH, xpath).text
        except:
            return ""

    def safe_attr(xpath, attr):
        try:
            return driver.find_element(By.XPATH, xpath).get_attribute(attr)
        except:
            return ""

    return {
        "State": state,
        "District": district,
        "Category": category,
        "Name": safe("//h1"),
        "Address": safe('//button[@data-item-id="address"]'),
        "Website": safe_attr('//a[@data-item-id="authority"]', "href"),
        "Phone": safe('//button[contains(@data-item-id,"phone")]')
    }


# =========================================================
# MAIN LOOP
# =========================================================
all_data = []
save_counter = 0

for state, districts in india_locations.items():

    for district in districts:

        for category in target_categories:

            query = f"{category} in {district}"

            try:
                print(f"\nSEARCHING: {query}")

                search_maps(query)

                listings = get_listings()

                print("FOUND:", len(listings))

                for i in range(len(listings)):

                    try:
                        listings = driver.find_elements(By.XPATH, '//a[contains(@href,"/place")]')
                        listings[i].click()
                        time.sleep(4)

                        data = extract_data(state, district, category)

                        if data["Name"].strip():
                            all_data.append(data)
                            save_counter += 1

                            print(data["Name"])

                            if save_counter % 50 == 0:
                                pd.DataFrame(all_data).to_csv("backup_india_leads.csv", index=False)

                    except Exception as e:
                        print("BUSINESS ERROR:", e)
                        continue

            except Exception as e:
                print("SEARCH ERROR:", e)

                try:
                    driver.quit()
                except:
                    pass

                driver = create_driver()
                wait = WebDriverWait(driver, 20)


# =========================================================
# SAVE FINAL FILE
# =========================================================
df = pd.DataFrame(all_data)

df.drop_duplicates(subset=["Name", "Phone"], inplace=True)
df = df[df["Name"].astype(str).str.strip() != ""]

df.to_csv("all_india_target_industries.csv", index=False)

print("\nDONE")
print("TOTAL RECORDS:", len(df))

driver.quit()